<a href="https://colab.research.google.com/github/tuankhoin/CO3133-Deep-Learning/blob/main/Week_7_Object_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Ho Chi Minh University of Technology (HCMUT)

CO3133 - Deep Learning and Its Applications

# Week 7 - Object Detection

### YOLO, Inference, Evaluation, and DETR

Classification asks **what is in the image**.  
Segmentation asks **what class is each pixel**.  
Object detection asks:

> **What objects exist, and where are they?**

This lecture develops object detection from outputs and labels to YOLO-style detectors, NMS, evaluation, and Transformer-based detection.

## Learning Outcomes

By the end of this lecture, you should be able to:

- Explain detection outputs and bounding-box formats.
- Explain **Backbone → Neck → Head**.
- Explain $P_3,P_4,P_5$ multi-scale features.
- Track YOLO tensor shapes.
- Explain reference points, box regression, and DFL.
- Explain one-to-many assignment and TaskAlignedAssigner.
- Describe YOLO loss components.
- Compare YOLOv8 and YOLO11 conceptually.
- Explain inference, confidence filtering, and NMS.
- Explain IoU, Precision, Recall, AP, and mAP.
- Explain DETR, object queries, and Hungarian matching.

## Contents

1. Detection Task and Labels
2. YOLO Architecture
3. Multi-Scale Features and Tensor Shapes
4. Detection Head and DFL
5. Assignment and Training Loss
6. YOLOv8 vs YOLO11
7. Inference and NMS
8. Evaluation
9. Data Pipeline and Common Problems
10. YOLOv26 and DETR
11. Hungarian Matching
12. Summary

---
# 1. Detection Task and Labels
---

## What Is Object Detection?

<img src="https://docs.microsoft.com/en-us/learn/wwl-data-ai/detect-objects-images-custom-vision/media/produce-objects.png" height=300/>

A detector predicts a variable-size set:

$$\boxed{
\hat Y=
\{(\hat b_i,\hat c_i,\hat s_i)\}_{i=1}^{N}}
$$

where:

- $\hat b_i$: bounding box;
- $\hat c_i$: class;
- $\hat s_i$: confidence;
- $N$: number of retained detections.

Detection therefore combines:

$$
\text{recognition}+\text{localization}.
$$

## Classification vs Detection vs Segmentation

| Task | Output | Spatial detail | Common metric |
|---|---|---|---|
| Classification | one label/image | none | Accuracy |
| Detection | boxes + labels | object-level | mAP |
| Segmentation | pixel labels/masks | pixel-level | mIoU / Dice |

Detection is useful when we need object location but not necessarily a label for every pixel.

## Why Detection Is Difficult

<img src="https://i.redd.it/kq9sbbgts5wd1.png" height=300/>

The model must handle:

- variable object counts;
- multiple object sizes;
- occlusion;
- background clutter;
- several predictions for one object;
- many background candidate locations.

This means object detection is not simply “classification with a box”.

## Bounding-Box Formats

### Corner representation

<img src="https://www.learnml.io/posts/a-guide-to-bounding-box-formats/index_files/figure-html/cell-17-output-1.png" height=400/>

$$\boxed{
(x_1,y_1,x_2,y_2)
}$$

Useful for IoU, clipping, and drawing.

### Center-size representation

<img src="https://dragoneye.ai/images/blog/a-guide-to-bounding-box-formats/image11.png" height=300/>

$$\boxed{
(c_x,c_y,w,h)
}$$

Useful for normalized labels and geometric transformations.

Conversion:

$$
x_1=c_x-\frac{w}{2},\qquad
x_2=c_x+\frac{w}{2},
$$

$$
y_1=c_y-\frac{h}{2},\qquad
y_2=c_y+\frac{h}{2}.
$$

## YOLO Label Format

A common YOLO annotation row is:

```text
class_id cx cy w h
```

with coordinates normalized to $[0,1]$.

Example:

```text
2 0.51 0.43 0.18 0.22
```

Typical pairing:

```text
001.jpg  ↔  001.txt
```

Each image may contain zero, one, or many objects.

## Dataset Organization

A common structure is:

```text
dataset/
    train/
        images/
        labels/
    val/
        images/
        labels/
    test/
        images/
        labels/
    data.yaml
```

`data.yaml` defines split paths and class names.

The most important rule:

> Keep train, validation, and test independent.

## Batched Ground-Truth Representation

Images have different object counts.

Let:

$$
N_{\max}
=
\max(\text{objects per image in the batch}).
$$

A padded batch may use:

$$
loc_{gt}\in\mathbb{R}^{B\times N_{\max}\times4},
$$

$$
cls_{gt}\in\mathbb{Z}^{B\times N_{\max}},
$$

and:

$$
mask\in\{0,1\}^{B\times N_{\max}}.
$$

Padding entries must be ignored during assignment and loss.

---
# 2. YOLO - You Only Look Once
---


<img src="https://raw.githubusercontent.com/tuankhoin/CO3057-Computer-Vision/refs/heads/main/assets/yolo1.gif" height=300/>

<img src="https://raw.githubusercontent.com/tuankhoin/CO3057-Computer-Vision/refs/heads/main/assets/yolo2.png" height=300/>

<img src="https://miro.medium.com/0*eFMc5GrSXAJ1gGPF.png" height=300/>

## (Not Examinable) Before YOLO: R-CNN

<img src="https://cdn.hashnode.com/res/hashnode/image/upload/v1710568978437/bcd29ce4-5733-4d09-8fc3-93c375b09a73.png" height=300/>

Steps were selecting multiple regions in the image -> Classify those regions.

## The YOLO way

<img src="https://miro.medium.com/v2/resize:fit:1400/0*QtUFUGVsSXwxR-u2" height=300/>

A one-stage YOLO detector can be viewed as:

$$
\boxed{
\text{Image}
\rightarrow
\text{Backbone}
\rightarrow
\text{Neck}
\rightarrow
\text{Head}
\rightarrow
\text{NMS}
\rightarrow
\text{Outputs}
}
$$

Rather than memorizing YOLO versions, learn what each module does.

## Backbone

The backbone extracts **multi-level visual features**.

For a $640\times640$ image, common feature levels are:

| Level | Stride | Map size |
|---|---:|---:|
| $P_3$ | 8 | $80\times80$ |
| $P_4$ | 16 | $40\times40$ |
| $P_5$ | 32 | $20\times20$ |
|...| s | $640/s$ |

Thus:

$$
P_3:(B,C_3,80,80),
$$

$$
P_4:(B,C_4,40,40),
$$

$$
P_5:(B,C_5,20,20).
$$

## Why Multiple Scales?

Fine/High spatial resolution:
- preserve local detail;
- useful for small objects.

Coarse/Lower spatial resolution:
- larger receptive field;
- stronger context;
- useful for large objects.

Therefore detection uses a **feature pyramid** rather than only the final CNN layer.

## Neck: Multi-Scale Feature Fusion

Low-level features contain detail but weak semantics.

High-level features contain context but weaker localization.

The neck combines them:

$$
P_3,P_4,P_5
\rightarrow
P_3^*,P_4^*,P_5^*.
$$

Common components:

- FPN (Feature Pyramid Network): top-down **semantic** propagation;
- PAN (Path Aggregation Network): bottom-up **positional/detail** aggregation.

## FPN and PAN Intuition

<img src="https://www.researchgate.net/publication/368474932/figure/fig2/AS:11431281120017067@1676347332188/The-structure-of-FPN-and-PAN-in-YOLOv5.png" height=300/>

- FPN:
$
P_5\rightarrow P_4\rightarrow P_3
$
: moves semantic information toward finer maps.
- PAN:
$
P_3\rightarrow P_4\rightarrow P_5
$: moves localization/detail information toward deeper maps.

The goal is for every scale to contain both:

- strong context;
- useful localization.

## Detection Head

The head predicts for every reference point:

1. box localization;
2. class logits.

Modern YOLO uses a decoupled idea:

$$\boxed{
\text{Feature}
\rightarrow
\begin{cases}
\text{box branch}\\
\text{class branch}
\end{cases}}
$$

Classification and localization therefore have separate final prediction paths.

---
# 3. Multi-Scale Features and Tensor Shapes
---

<img src="https://cdn.inflearn.com/public/files/posts/73495669-f634-44cf-94ec-e44648c70e58/%E1%84%89%E1%85%B3%E1%84%8F%E1%85%B3%E1%84%85%E1%85%B5%E1%86%AB%E1%84%89%E1%85%A3%E1%86%BA%202021-04-14%20%E1%84%8B%E1%85%A9%E1%84%92%E1%85%AE%203.13.28.png" height=500/>

## Reference Points

<img src="https://raw.githubusercontent.com/tuankhoin/CO3057-Computer-Vision/refs/heads/main/assets/detectionhead.png" height=300/>

Each feature-map cell corresponds to a reference point in the input image.

For level $m$ with stride $s_m$:

$$
x_c=(j+0.5)s_m,
$$

$$
y_c=(i+0.5)s_m.
$$

The model predicts box distances relative to this location.

## Number of Dense Prediction Locations

For input $640\times640$:
$
N_a
=
80^2+40^2+20^2
=
8400$

Thus YOLO makes predictions at thousands of spatial locations before filtering. This is called **dense prediction**.

## Head Tensor Shapes

Let:

- $N_a$: total reference locations;
- $N_c$: classes;
- $N_r$: DFL (Distribution Focal Loss) bins.

Then:

- $
loc\in\mathbb{R}^{B\times N_a\times4N_r}
$
- $
cls\in\mathbb{R}^{B\times N_a\times N_c}
$

- After decoding and NMS:
$
output\in\mathbb{R}^{B\times N_{det}\times6}.
$
- Typical final fields:
$
(x_1,y_1,x_2,y_2,score,class)$

## Flattening $P_3$,$P_4$,$P_5$

Let:
$
n_3=H_3W_3,\quad
n_4=H_4W_4,\quad
n_5=H_5W_5.
$

Then:
$\boxed{
N_a=n_3+n_4+n_5}
$

One common ordering is:
$
P_3\rightarrow P_4\rightarrow P_5.
$

A global prediction index can therefore be mapped back to one feature level and one grid cell.

## Mapping Back to the Original Image

If the input is letterboxed with scale factor $r$ and padding $(d_x,d_y)$:

$$
x_{orig}
=
\frac{x-d_x}{r},
$$

$$
y_{orig}
=
\frac{y-d_y}{r}.
$$

This inverse transform is essential during inference.

---
# 4. Detection Head and DFL
---

<img src="https://raw.githubusercontent.com/tuankhoin/CO3057-Computer-Vision/refs/heads/main/assets/detection2.png" height=300/>

## Class Prediction

For every reference point, the class branch outputs:
$
N_c
$
logits.

In YOLOv8/YOLO11-style training:

- sigmoid converts logits to independent class probabilities;
- BCE is used for classification.

At inference:

- weak boxes are filtered by confidence;
- the strongest class is selected.

## Box Prediction

Modern YOLO predicts four distances: left, top, right, bottom

Instead of predicting one scalar directly, each side can be represented as a probability distribution over:

$$
0,1,\ldots,N_r-1.
$$

This is the basis of **Distribution Focal Loss (DFL)**.

## DFL Decoding

<img src="https://velog.velcdn.com/images/d4r6j/post/26ffd850-8ea0-40df-a4b4-51e0053be4df/image.png" height=300/>

If one side predicts probabilities:

$$
p_0,\ldots,p_{N_r-1},
$$

the predicted continuous distance is:

$$
\hat d
=
\sum_{k=0}^{N_r-1}
k\,p_k.
$$

So discrete bins are converted back into a continuous box offset.

## DFL Soft Target

<img src="https://raw.githubusercontent.com/tuankhoin/CO3057-Computer-Vision/refs/heads/main/assets/dflsoft.png" height=250/>

Suppose:

$$
y=2.45.
$$

Then:

$$
y^-=2,\qquad
y^+=3.
$$

Weights are:

$$
w^-=y^+-y=0.55,
$$

$$
w^+=y-y^-=0.45.
$$

Instead of assigning the target entirely to bin 2 or 3, DFL spreads supervision between both neighbors.

---
# 5. Assignment and Training Loss
---

## Why Assignment Is Necessary

- The detector may produce:
$
N_a=8400
$
candidate positions while the image has only a few objects.
- Training must decide: Which candidate positions should learn from each ground-truth box?

That step is called **assignment**.

## One-to-Many Assignment

Define:
$
m_{ik}\in\{0,1\}
$
for prediction $i$ and ground-truth object $k$.

One-to-many assignment allows:
$
\sum_i m_{ik}>1$

So one object may supervise several predictions.

Benefits:

- more positive gradients;
- faster learning;
- strong recall.

Cost:

- duplicate predictions can appear.

## TaskAlignedAssigner

A simplified alignment score is:

$$\boxed{
t_{ik}
=
\sigma(z_{i,c_k})^\alpha
\cdot
IoU(b_i,b_k^{gt})^\beta}
$$

A good positive candidate therefore has both:

- strong class confidence;
- strong localization overlap.

For each GT, top candidates are selected as positives.

## YOLO Loss

After assignment:

$$\boxed{
\mathcal{L}
=
w_{box}\mathcal{L}_{box}
+
w_{cls}\mathcal{L}_{cls}
+
w_{dfl}\mathcal{L}_{dfl}}
$$

- Box loss:
Localization quality, often IoU/CIoU-based.
- Classification loss:
BCE over class logits.
- DFL:
Distribution-based supervision of box-side distances.

## Classification Loss

With:

$$
\hat p_{i,c}=\sigma(z_{i,c}),
$$

a BCE form is:

$$\boxed{
\mathcal{L}_{cls}
=
-\frac{1}{N_a}
\sum_i\sum_c
\left[
y_{i,c}\log\hat p_{i,c}
+
(1-y_{i,c})\log(1-\hat p_{i,c})
\right]}
$$

Background cells use zero class targets.

## Localization Loss

Only positively assigned candidates have ground-truth boxes.

A conceptual box term is:

$$\boxed{
\ell_{box}
=
1-CIoU(b_i,b_k^{gt})}
$$

Therefore:

- positive cells contribute to box/DFL loss;
- background cells mainly contribute classification/background supervision.

## DFL Loss

For one side:

$$\boxed{
\mathcal{L}_{DFL}
=
-
\left(
w^-\log \hat p_{y^-}
+
w^+\log \hat p_{y^+}
\right)}
$$

The procedure is repeated for:
$
l,t,r,b.
$

---
# 6. YOLOv8 vs YOLO11
---

<img src="https://miro.medium.com/v2/resize:fit:1400/1*U9sZWk80XfWmfUlxnVyd0g.png" height=250/>

## Compare Versions by Function

Do not memorize version names first.

Compare:

1. Backbone
2. Neck
3. Head
4. Box regression
5. Classification

The source lecture emphasizes that YOLOv8 and YOLO11 share the same basic detection philosophy.

## YOLOv8

Components:
- Conv: reduces spatial resolution and increases the number of feature channels.
- C2f: a CSP-style block that improves representation power while keeping
computational cost reasonable.
- SPPF: aggregates context at deep layers using fast pooling.


<img src="https://media.springernature.com/lw685/springer-static/image/chp%3A10.1007%2F978-981-99-7962-2_39/MediaObjects/604499_1_En_39_Fig2_HTML.png" height=300/>

### Backbone

- Conv
- C2f
- SPPF

### Neck

- FPN/PAN
- C2f refinement

### Head

- decoupled box/class branches

### Prediction

- DFL for box regression
- sigmoid + BCE for classes

## YOLO11

Components:
- C3k2: replaces the role of C2f, aiming for more efficient representation.
- SPPF: still used to aggregate context at deep layers.
- C2PSA: adds an attention mechanism to improve feature selectivity.


<img src="https://arxiv.org/html/2604.03349v1/YOLOv11_arch.png" height=400/>

### Backbone

- Conv
- **C3k2**
- SPPF
- **C2PSA**

### Neck

- FPN/PAN
- **C3k2 refinement**

### Head (fully changed)

- same broad box/class split
- optimized feature processing, including DW/PW-style convolution

### Prediction: The same

The main changes are in feature processing efficiency and representation.

## YOLOv8 vs YOLO11 Summary

| Component | YOLOv8 | YOLO11 |
|---|---|---|
| Backbone | C2f + SPPF | C3k2 + SPPF + C2PSA |
| Neck | FPN/PAN + C2f | FPN/PAN + C3k2 |
| Head philosophy | decoupled | decoupled |
| Box regression | DFL | DFL |
| Classification | sigmoid + BCE | sigmoid + BCE |

### Key takeaway

YOLO11 changes **how features are processed**, not the basic detection formulation.

## Academic Modules vs Ultralytics YAML

Conceptually:

$$
\text{Backbone}
\rightarrow
\text{Neck}
\rightarrow
\text{Detection Head}.
$$

In Ultralytics YAML, the section named `head:` can contain both:

- neck/fusion operations;
- final Detect layers.

So YAML names should be mapped back to the academic functional modules.

---
# 7. Inference and NMS
---

## Inference Pipeline

A typical YOLO inference process is:

$$
\text{Image}
\rightarrow
\text{Resize/Letterbox}
\rightarrow
\text{Forward}
\rightarrow
\text{Decode Boxes}
\rightarrow
\text{Confidence Filter}
\rightarrow
\text{NMS}
\rightarrow
\text{Final Detections}.
$$

Inference therefore includes both the neural network and post-processing.

## Confidence Filtering

The head initially outputs many candidates.

Before NMS:

1. convert logits to scores;
2. discard boxes below a confidence threshold;
3. optionally limit the maximum candidate count.

This removes obvious background predictions.

## Non-Maximum Supression (NMS)

<img src="https://cdn.analyticsvidhya.com/wp-content/uploads/2020/07/graphic4.jpg" height=300/>

Dense one-to-many detectors may output several overlapping boxes for the same object.

NMS attempts to keep only the strongest one.

Core idea:

> High confidence wins; highly overlapping weaker boxes are removed.

## Greedy NMS

For each class:

1. sort boxes by confidence;
2. keep the highest-scoring box;
3. compare it with all remaining boxes;
4. suppress boxes when:

$$
IoU>\tau;
$$

5. repeat.

The IoU threshold $\tau$ controls how aggressively overlap is suppressed.

## NMS Example

Suppose:

$$
A_1=0.95,\quad
A_2=0.82,\quad
A_3=0.74.
$$

If:

$$
IoU(A_1,A_2)=0.68>0.5,
$$

$$
IoU(A_1,A_3)=0.73>0.5,
$$

then $A_2$ and $A_3$ are removed after keeping $A_1$.

NMS is duplicate removal, not classification.

---
# 8. Evaluation
---

## Evaluation Roadmap

1. Match predictions to ground truth using IoU.
2. Count TP / FP / FN.
3. Compute Precision / Recall.
4. Sweep confidence thresholds.
5. Build the PR curve.
6. Compute AP.
7. Average over classes → mAP.

Detection performance depends on both:

$$
\text{prediction quality}
+
\text{localization quality}.
$$

## IoU

<img src="https://b2633864.assetcdn.net/2633864/wp-content/uploads/2016/09/iou_equation.png?lossy=2&strip=1&webp=1" height=300/>

For predicted box $B_{pred}$ and ground-truth box $B_{gt}$:

$$
IoU
=
\frac{
|B_{pred}\cap B_{gt}|
}{
|B_{pred}\cup B_{gt}|
}.
$$

Higher IoU means stronger localization overlap.

A prediction usually becomes a match only if:

- class is correct;
- $IoU\ge\tau$.

## TP, FP, FN

### True Positive

Correct class + sufficient IoU.

### False Positive

Examples:

- wrong class;
- poor box localization;
- duplicate box.

### False Negative

A ground-truth object is not successfully detected.

## Precision and Recall

$$
Precision
=
\frac{TP}{TP+FP}.
$$

> Of all predictions, how many were correct?

$$
Recall
=
\frac{TP}{TP+FN}.
$$

> Of all real objects, how many were found?

High precision means fewer false alarms.

High recall means fewer missed objects.

## Confidence Threshold Trade-Off

Higher confidence threshold:

- removes more weak predictions;
- often increases precision;
- may reduce recall.

Lower threshold:

- keeps more predictions;
- often increases recall;
- may reduce precision.

Therefore one threshold cannot summarize detector quality.

## Precision-Recall Curve

Sweep confidence thresholds.

Each threshold gives:

$$
(Recall,Precision).
$$

Connect these points to form the PR curve.

A better detector keeps Precision high over a wide Recall range.

## Average Precision

<img src="https://forums.fast.ai/uploads/default/original/2X/2/27a3e0be4fcce1b8b6dd534ec57efe3955064e94.png" height=300/>

<img src="https://cdn.learnopencv.com/wp-content/uploads/2022/08/04103952/mean-average-precision-map-calculation-11-point-interpolation-pascal-voc-manual.gif" height=300/>

For one class:

$$
AP
=
\int_0^1 P(R)\,dR.
$$

AP summarizes the area under the PR curve.

Examples:

- $AP_{50}$: IoU threshold $0.50$;
- $AP_{75}$: IoU threshold $0.75$.

A stricter IoU threshold demands more accurate localization.

## Mean Average Precision


For $N_c$ classes:

$$
mAP
=
\frac{1}{N_c}
\sum_{c=1}^{N_c}
AP_c.
$$

Common reports:

- `mAP@50`
- `mAP@50:95`

`mAP@50:95` evaluates performance across **multiple IoU thresholds**.

## Evaluation Can Be Wrong Even When the Formula Is Right

Common problems:

- inconsistent resize settings;
- incorrect class IDs;
- train/validation leakage;
- different NMS settings;
- poor annotations;
- incorrect box conversion.

Therefore:

> Bad evaluation protocol → bad conclusion.

---
# 9. Data Pipeline and Common Problems
---

## Common Detection Formats

### YOLO
```text
class cx cy w h
```

### COCO
JSON annotations.

### Pascal VOC
XML annotations.

Correct conversion matters more than the original annotation format.

## Small Objects and Rare Classes

<img src="https://kenh14cdn.com/203336854389633024/2026/2/6/vinfast-vf3-plus-1-1770341493941-17703414945011886415748.jpg" height=200/>

Small objects contain few pixels.

Possible responses:

- increase resolution;
- use fine feature levels;
- targeted augmentation;
- improve multi-scale fusion;
- collect more examples.

Rare classes may require:

- more data;
- resampling;
- focused augmentation.

Architecture cannot fully compensate for missing data.

## Detection Augmentation

<img src="https://www.googleapis.com/download/storage/v1/b/kaggle-forum-message-attachments/o/inbox%2F4692121%2Fa41aae822eefb59a838b09a2d29b335e%2FDA.jpg?generation=1722610941371695&alt=media" height=300/>

Common examples:

- flip;
- HSV changes;
- Mosaic;
- MixUp.

Geometric augmentation must update both:

- image;
- bounding boxes.

Otherwise supervision becomes incorrect.

## Data Throughput

A slow loader can make the GPU wait.

Possible optimizations:

- more workers;
- cache;
- faster storage.

Training speed depends on:

$$
\text{model speed}
+
\text{data pipeline speed}.
$$

---
# 10. YOLOv26 and DETR
---

<img src="https://arxiv.org/html/2601.12882v1/dfl.png" height=300/>

## Why Modern Detectors Move Toward One-to-One Prediction

Traditional YOLO:

- one object may supervise several candidates;
- several predictions may survive before NMS.

This gives useful dense supervision but creates duplicate outputs.

Modern methods explore:

> Can training itself encourage one clean prediction per object?

## YOLOv26 Dual-Head Idea

The source lecture presents YOLOv26 as:

$$
\text{Input}
\rightarrow
\text{Backbone}
\rightarrow
\text{Neck}
\rightarrow
\text{Dual Detection Heads}.
$$

Two branches:

1. one-to-many;
2. one-to-one.

Both can use the same neck features.

## Role of the Two Heads

### One-to-Many

- multiple positives per GT;
- strong dense supervision;
- good recall.

### One-to-One

- one best prediction per GT;
- cleaner output;
- fewer duplicates;
- less reliance on NMS.

Training can exploit both behaviors.

## Same Architecture ≠ Same Training Target

The heads can have very similar network structure but different:

- assignment strategy;
- positive selection;
- supervision;
- inference usage.

Thus:

$$
\boxed{
\text{same architecture}
\neq
\text{same learned behavior}
}
$$

## DETR: Detection Transformer

<img src="https://miro.medium.com/max/1400/1*rJ5rPa5JHVy297wbJ23Atg.png" height=300/>

DETR predicts a set of objects using:

$$
\text{CNN Features}
\rightarrow
\text{Transformer Encoder}
\rightarrow
\text{Object Queries}
\rightarrow
\text{Transformer Decoder}
\rightarrow
\text{Class + Box}.
$$

Unlike dense YOLO grids, DETR uses a smaller fixed number of learned object queries.

## Link Back to Week 5

Image features
$
I\in\mathbb{R}^{B\times C\times H\times W}
$
are flattened:

$$
Flatten(I)\in\mathbb{R}^{B\times HW\times C}.
$$

Add positional information:

$$
X=Flatten(I)+PE.
$$

Then the Transformer encoder mixes global context.

This is the same sequence-processing idea from Week 5, applied to visual features.

## Object Queries

DETR has learned queries:

$$
Q_0\in\mathbb{R}^{N_q\times C}.
$$

After batch expansion:

$$
Q_0\in\mathbb{R}^{B\times N_q\times C}.
$$

The decoder computes:

$$
Q_f=Decoder(Q_0,F,F).
$$

Each final query embedding represents one candidate object.

## DETR Outputs

Each query predicts:

- one class;
- one normalized box;
- or no-object.

Shapes:

$$
\hat p
\in
\mathbb{R}^{B\times N_q\times(N_c+1)},
$$

$$
\hat b
\in
\mathbb{R}^{B\times N_q\times4}.
$$

The extra class represents no-object.

## YOLO vs DETR

| Property | YOLO-style | DETR |
|---|---|---|
| Candidate source | dense spatial locations | learned queries |
| Number of candidates | thousands | fixed smaller set |
| Assignment | one-to-many | one-to-one |
| Duplicate predictions | common before NMS | reduced by training |
| NMS | traditionally used | usually unnecessary |

---
# 11. Hungarian Matching
---

<img src="https://www.thinkautonomous.ai/blog/content/images/2023/02/graph-lidar-camera.jpg" height=300/>

## Why Matching Is Needed

Ground truth:

$$
Y=\{y_i\}_{i=1}^{N_{gt}}
$$

Predictions:

$$
\hat Y=\{\hat y_j\}_{j=1}^{N_q}.
$$

DETR must decide:

> Which query should be responsible for each ground-truth object?

Because predictions form an unordered set, this cannot be solved by fixed index positions.

<img src="https://raw.githubusercontent.com/tuankhoin/CO3057-Computer-Vision/refs/heads/main/assets/hungarian.png" height=200/>

## Matching Cost

For ground-truth object $i$ and query $j$:

$$
C_{ij}
=
C^{cls}_{ij}
+
\lambda_1C^{L1}_{ij}
+
\lambda_gC^{giou}_{ij}.
$$

A conceptual form is:

$$
C_{ij}
=
-\log \hat p_j(c_i)
+
\lambda_1\|b_i-\hat b_j\|_1
+
\lambda_g\mathcal{L}_{GIoU}(b_i,\hat b_j).
$$

**Lower cost means a better match.**

## One-to-One Constraint

If every ground-truth object independently chose its cheapest query, two GT objects might select the same query.

Hungarian matching enforces:

- each GT → exactly one query;
- each query → at most one GT.

The assignment must therefore be globally consistent.

## Globally Optimal Assignment

Hungarian matching finds:

$$
\sigma^*
=
\arg\min_\sigma
\sum_{i=1}^{N_{gt}}
C_{i,\sigma(i)}.
$$

The goal is not to choose the cheapest element in every row independently.

The goal is to minimize the **total assignment cost** under one-to-one constraints.

## Loss After Matching

For matched pairs:

$$
\mathcal{L}
=
\mathcal{L}_{cls}
+
\lambda_1
\|b-\hat b_{\sigma^*}\|_1
+
\lambda_g
\mathcal{L}_{GIoU}.
$$

Unmatched queries are trained as no-object.

This one-to-one supervision is why DETR can usually avoid YOLO-style NMS.

## YOLO Assignment vs Hungarian Matching

| Property | Dense YOLO | DETR |
|---|---|---|
| Assignment | one-to-many | one-to-one |
| Selection | task-aligned/local | Hungarian/global |
| Positives per GT | several | one |
| Unselected candidates | background | no-object |
| Duplicate removal | NMS | reduced by matching |

This is one of the central conceptual differences in modern detection.

---
# 12. Summary
---

## Object Detection

A detector predicts:

$$
\hat Y=
\{(\hat b_i,\hat c_i,\hat s_i)\}_{i=1}^{N}.
$$

Main challenges:

- localization;
- classification;
- multiple scales;
- variable object counts;
- duplicate predictions.

## YOLO Mental Model

Always start with:

$$
\boxed{
\text{Backbone}
\rightarrow
\text{Neck}
\rightarrow
\text{Head}
}
$$

Backbone:

$$
P_3,P_4,P_5
$$

Neck:

$$
P_3,P_4,P_5
\rightarrow
P_3^*,P_4^*,P_5^*
$$

Head predicts box distributions and class logits.

## Inference Summary

Typical YOLO inference:

$$
\text{Image}
\rightarrow
\text{Forward}
\rightarrow
\text{Decode}
\rightarrow
\text{Confidence Filter}
\rightarrow
\text{NMS}
\rightarrow
\text{Final Detections}.
$$

NMS removes strongly overlapping duplicates.

## Evaluation Summary

$$
IoU
\rightarrow
TP/FP/FN
\rightarrow
Precision/Recall
\rightarrow
PR\ Curve
\rightarrow
AP
\rightarrow
mAP.
$$

A detector is not good merely because it predicts the correct class.

Its boxes must also localize objects accurately.

## Version Summary

### YOLOv8
C2f-based feature processing, FPN/PAN, decoupled head, DFL.

### YOLO11
Same broad detection philosophy with C3k2/C2PSA and more efficient refinement.

### YOLOv26 direction
One-to-many + one-to-one supervision.

### DETR
Transformer queries + Hungarian matching + set-based prediction.

## How to Read Detection Code

When studying an implementation, inspect:

1. model/YAML graph;
2. backbone and neck blocks;
3. detection head;
4. assignment code;
5. loss functions;
6. box decoding;
7. NMS or matching;
8. mAP evaluation.

The final goal is not only to run YOLO, but to understand how to modify and design detectors.

## Week 7 Checklist

You should now be able to answer:

1. What does an object detector output?
2. How do `xyxy` and `cxcywh` differ?
3. Why use $P_3,P_4,P_5$?
4. What do Backbone, Neck, and Head do?
5. Why are there thousands of YOLO candidate locations?
6. How does a feature-map cell map to image coordinates?
7. What does the class branch predict?
8. What is DFL?
9. Why does one GT supervise multiple YOLO predictions?
10. What does TaskAlignedAssigner combine?
11. What are the main YOLO loss components?
12. How do YOLOv8 and YOLO11 differ?
13. Why is NMS necessary?
14. What are TP, FP, and FN in detection?
15. How do Precision and Recall differ?
16. What is AP? What is mAP?
17. Why can evaluation settings change mAP?
18. What is an object query in DETR?
19. Why does DETR use Hungarian matching?
20. What is the key difference between YOLO one-to-many and DETR one-to-one training?